# The Immigrant Renter Penalty

**DS4DH · Module 02 — Describing and Comparing Data**

*Technique:* Ranking responsibly, and converting percentage points into dollars

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/02d_immigrant_penalty.ipynb)

Data: `merged_dataset.csv`, `top10_immigrant_penalty.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This notebook reads the CSVs sitting next to it. In Colab, upload them from
# the pack's data/ folder when prompted. The exists() guard means a re-run
# part-way through a session will not ask you to upload all over again.
NEEDED = ['merged_dataset.csv', 'top10_immigrant_penalty.csv']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    try:
        from google.colab import files
        print('Upload from the data/ folder of the pack: ' + ', '.join(missing))
        files.upload()
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))

df       = pd.read_csv('merged_dataset.csv')
df_top10 = pd.read_csv('top10_immigrant_penalty.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Two things that turn a statistic into something a reader can act on: ranking, and
converting to money.

Both are also the two easiest places to mislead. A ranking of small samples is
mostly a ranking of noise, and a dollar conversion inherits every assumption in
the denominator you divide by.

In [ ]:
csd = df.dropna(subset=['csd_code'])
imm = csd[csd['immigrant_status'] == 'Immigrant'][
    ['csd_code', 'geography_name', 'cma', 'Renter', 'rent_income']]
nim = csd[csd['immigrant_status'] == 'Non-immigrants'][['csd_code', 'Renter']]
pen = imm.merge(nim, on='csd_code', suffixes=('_imm', '_nim')).dropna(
    subset=['Renter_imm', 'Renter_nim'])
pen = pen[pen['cma'].isin(CITIES)].copy()
pen['premium'] = pen['Renter_imm'] - pen['Renter_nim']

print(f'{len(pen)} CSDs with both groups reported')
print(f'positive premium (immigrants pay more) in {(pen["premium"] > 0).sum()} of them')
print(f'negative premium (immigrants pay less) in {(pen["premium"] < 0).sum()} of them')

That split is the first honest finding: the penalty is **not** universal. It runs
in both directions depending on the municipality. Any headline saying "immigrant
renters pay more" is describing a majority tendency, not a rule.

In [ ]:
# The published top-10 table ships with the pack. Check it reproduces.
top10 = pen.nlargest(10, 'premium')[
    ['geography_name', 'cma', 'Renter_imm', 'Renter_nim', 'premium']]
print('Recomputed top 10 by premium:')
print(top10.to_string(index=False))
print()
print('Shipped file for comparison:')
print(df_top10.to_string(index=False))

### 🔧 Your turn 1

Compare the two tables above row by row.

Do they match? If a row differs, which of the filtering decisions in the first
cell would explain it?

## Ranking responsibly

A league table implies the top entry is the worst case. With small denominators
it often just means the noisiest case.

In [ ]:
# Attach the renter population behind each premium.
sized = pen.merge(
    csd[csd['immigrant_status'] == 'Immigrant'][['csd_code', 'rent_pop']],
    on='csd_code', how='left')

print(f'{"Geography":<34}{"CMA":<11}{"premium":>9}{"renter hh":>11}')
print('-' * 65)
for _, r in sized.nlargest(10, 'premium').iterrows():
    pop = f'{r["rent_pop"]:,.0f}' if pd.notna(r['rent_pop']) else 'n/a'
    print(f'{r["geography_name"][:33]:<34}{r["cma"]:<11}{r["premium"]:>+9.1f}{pop:>11}')
print()
print('Read the right-hand column before the left one.')

In [ ]:
# Re-rank with a minimum size filter and watch the table change.
MIN_RENTER_HH = 200

big = sized[sized['rent_pop'] >= MIN_RENTER_HH]
print(f'CSDs with at least {MIN_RENTER_HH} renter households: {len(big)} of {len(sized)}')
print()
print(f'{"Geography":<34}{"CMA":<11}{"premium":>9}{"renter hh":>11}')
print('-' * 65)
for _, r in big.nlargest(8, 'premium').iterrows():
    print(f'{r["geography_name"][:33]:<34}{r["cma"]:<11}'
          f'{r["premium"]:>+9.1f}{r["rent_pop"]:>11,.0f}')

### 🔧 Your turn 2

Change `MIN_RENTER_HH` to 50, then 500, re-running each time.

How much of the original top 10 survives at 500? A ranking that dissolves under a
size filter was never a ranking of places — it was a ranking of sample sizes.

## Percentage points into dollars

A percentage point is a share of income, so converting it needs an income. Which
income you pick changes the answer, and the reader cannot see your choice unless
you state it.

In [ ]:
base_inc = pen['rent_income'].median()
print(f'Median immigrant renter household income: ${base_inc:,.0f}')
print()
print(f'{"Premium (pp)":>13}{"Extra per year":>18}{"Per month":>13}')
print('-' * 44)
for pp in [1, 2, 5, 10, 16]:
    yearly = base_inc * pp / 100
    print(f'{pp:>13}{yearly:>18,.0f}{yearly / 12:>13,.0f}')
print()
print('The largest premium in the table is about 16pp. At this income that is')
print(f'roughly ${base_inc * 0.16:,.0f} a year — the scale a programme must match.')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
show = big.nlargest(12, 'premium').sort_values('premium')
labels = [f'{n[:26]} ({c[:3]})' for n, c in
          zip(show['geography_name'], show['cma'])]
ax.barh(labels, show['premium'], color='#E8663D')
ax.set_xlabel('Immigrant renter premium (percentage points)')
ax.set_title(f'Largest premiums among CSDs with ≥{MIN_RENTER_HH} renter households')
plt.tight_layout()
plt.show()

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** The shipped `top10_immigrant_penalty.csv` matches the recomputed
table. It was produced by the same filtering: drop aggregate rows, keep the four
CMAs, require both groups reported. Where a row looked different it is usually
the `Canada` or CMA-level rows leaking in when `csd_code` is not dropped first.

**Your turn 2.** At `MIN_RENTER_HH = 500` most of the original top 10 disappears.
The extreme premiums live in municipalities with a few dozen renter households,
where a handful of households sets the published rate. This is why the course
moves to significance testing next: the eye-catching entries in a league table
are frequently the ones with the least evidence behind them.

Note also that the dollar conversion uses the *median immigrant renter income*.
Using each CSD's own income would give a different — and defensibly better —
figure per row. The single-income version is a communication simplification, and
it should be labelled as one.

</details>

## Where this stops

You can rank, filter and monetise. You still cannot say a single one of these
premiums is distinguishable from zero.

Next: **04a — the t-test**, which is the first tool in the pack that answers that.